In [8]:
import ee
ee.Authenticate(auth_mode="gcloud")
ee.Initialize(project= 'rmrs-wildfire-treatments')

import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
from datetime import datetime
from datetime import timedelta
import os
import sys
from datetime import datetime
import math
import sklearn
from sklearn.preprocessing import StandardScaler
import importlib
import re
import glob
from pathlib import Path
from google.cloud import storage

In [9]:
# import custom functions
module_path = Path("function_scripts/gee").resolve()
sys.path.append(str(module_path))

import mask_function
importlib.reload(mask_function)
import burn_severity_veg_indices
importlib.reload(burn_severity_veg_indices)

import landcover_functions
import topography_functions
import snow_functions
import gcs_functions
import daily_weather
importlib.reload(daily_weather)

import climate_functions
importlib.reload(climate_functions)
import simple_extract_function
importlib.reload(simple_extract_function)



<module 'simple_extract_function' from 'C:\\Users\\HannahVanDusen\\Desktop\\WildfireTreatmentOutcomes\\scripts\\klamath\\function_scripts\\gee\\simple_extract_function.py'>

In [3]:
!gcloud auth login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=Tlz0MMxT2xBVvuwJKEc5dYPwp1kR3Y&access_type=offline&code_challenge=ucgxVJL_hFJb-d_VkyM_zEErzzBanNUm4cjmMHtXfgs&code_challenge_method=S256


You are now logged in as [hannah.vandusen@usda.gov].
Your current project is [rmrs-wildfire-treatments].  You can change this setting by running:
  $ gcloud config set project PROJECT_ID


In [10]:
# import user-defined settings

# get pathfile of this script
import ipynbname
notebook_path = ipynbname.path()
project_root = notebook_path.parent

import pandas as pd
import ast

# Full local path to user-input csv file
input_path = os.path.join(project_root, "user_input/user_input_general.csv")

# read user-input CSV
df = pd.read_csv(input_path)

# container for created objects from user input csv
user_inputs = {}

for _, row in df.iterrows():
    # Skip rows flagged as R expressions 
    if row["treat_as_R_expression"]: 
        continue
    
    raw = row["value"]

    # Try safe literal parsing; fall back to raw string 
    try: 
        val = ast.literal_eval(raw) 
    except Exception: 
        val = raw 
        
    user_inputs[row["name"]] = val

In [17]:
# User Input parameters for Burn Severity:

# 1) WTO outcome 
wto_outcome = 'burn_severity'

# 2) Directories 
# local directory (if using the WTO box only change your name)
local_directory = user_inputs["processed_data_directory"]

# Location of sampling points or shapes
# *important* sampling points or shape must have a column "point_id" which is is unique value for each point per fire
# this allows for the joining of gee data and local data (this column will not be included in the final dataframe) 

resol = 270
point_directory = local_directory + '/initial_sampling_points/' + wto_outcome 
filename_suffix =  '_initial_pts'


# 3) File naming conventions 
# Google Cloud Buckets and directory 
bucket_name = 'bb-gee-bucket' 
bucket_fp = 'klamath/processed_outputs' # to create this folder, must go to https://console.cloud.google.com/storage/browser/bb-gee-bucket/ and add new folder

# GEE output: file prefix for google cloud bucket and local computer 
gee_output_subdirectory = '' # For infra only: '/simple_pt', '/buff_30m', or '/buff_100m'
gee_output_directory = wto_outcome + '/gee_point_extracts' + gee_output_subdirectory

# make directory
os.makedirs(user_inputs["processed_data_directory"] + "/processed_outputs/" + gee_output_directory, exist_ok=True)
print("gee output directory: " + gee_output_directory)

# 4) GEE reducer type for all bands
reducer = ee.Reducer.median()

# 5) List of Fires 
fire_list = pd.read_csv(local_directory + '/fire_lists/' + wto_outcome + '_fire_list.csv')

# subset fires to only those that have burn progression maps (some are too small to get them)
fire_progression_path = "date_of_burn/dob_tif"
fire_progression_folder_path = os.path.join(local_directory, fire_progression_path)

fire_progression_files = [
    f for f in os.listdir(fire_progression_folder_path)
    if f.endswith('_1_dob.tif') 
]

fire_progression_names = [f.split('_', 1)[0] for f in fire_progression_files]


# load variable crosswalks
variable_list = pd.read_csv(user_inputs["global_data_directory"] + "/variable_descriptions/variable_crosswalks_simple.csv")

# 6) Do you want to use the different mask functions
# * you can turn on/off which masks in the 'variable_crosswalks_simple' file 
mask = True

# Secondary mask parameter in sef.process_fire_data function 
mask2 = False 

# do you want to buffer your points
buffer = False
buffer_size = 100


# Add year to fire list
fire_list['Year'] = pd.to_datetime(fire_list['Ignition']).dt.year
fire_list = fire_list[fire_list['MTBS_ID'].isin(fire_progression_names)]
#print(fire_list)

gee output directory: burn_severity/gee_point_extracts


In [21]:
# Fires of interest
#fire_list2 = fire_list[fire_list["Year"] == 2018]
fireIDs = fire_list['MTBS_ID'].values.tolist()
#fireIDs = ['CA4075212333720210731']

# empty list of processed fires
processed_fires = []

#for fireID in fireIDs: 
for fireID in fireIDs:  
        
     # List all files of sampling points for each file
    # Large fires will have muliple files 
    pattern = os.path.join(point_directory, f"{fireID}_*.csv")
    files = glob.glob(pattern)

    # If no files exist then skip 
    if len(files) == 0:
        print(f"Skipping {fireID} fire: Shapefile not found at {point_directory}")
     

    elif len(files) > 0:
        if sum(fire_list['MTBS_ID'] == fireID) == 0: continue

        for i, file in enumerate(files):
            # Read in CSV of points with XY coordinates
            df = pd.read_csv(files[0])
            shape_gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.X, df.Y, crs=user_inputs["CRS"]))

            # pare down columns (causing issues when too big, and not necessary for extraction)
            cols_to_remove = ["latitude", "longitude", "lat", "long"]

            # Drop only the columns that actually exist
            shape_gdf = shape_gdf.drop(columns=[c for c in cols_to_remove if c in shape_gdf.columns])
            
            # Buffer if necessary
            if buffer:
                shape_gdf["geometry"] = shape_gdf["geometry"].buffer(buffer_size)
                wto_outcome2 = wto_outcome + '_buff'
            else:
                wto_outcome2 = wto_outcome
            
            # Convert to Earth Engine FeatureCollection
            shape_feat = geemap.geopandas_to_ee(shape_gdf)
            if mask:
                # Create a latitude and longitude image for masking points
                latlong = ee.Image.pixelLonLat()
                latlong_mask = mask_function.mask_function(latlong, local_directory, fireID, fire_list, variable_list, wto_outcome)
                shapes_latlong = latlong_mask.reduceRegions(collection = shape_feat, reducer = ee.Reducer.first(), scale = 30)
                shape_feat = shapes_latlong.filter(ee.Filter.neq('latitude', None))
            
            # Check if mask image has any valid pixels
            mask_stats = latlong_mask.mask().reduceRegion(
                reducer=ee.Reducer.sum(),
                geometry=shape_feat.geometry(),
                scale=30,
                maxPixels=1e9
            ).getInfo()

            # If mask values are None or sum is 0, skip this fire
            if not mask_stats or all(v is None or v == 0 for v in mask_stats.values()):
                print(f"Skipping fire {fireID}: mask image is fully masked.")
                continue


            if i == 0:
                filename_suffix2 = filename_suffix
            else:
               filename_suffix2 = '_' + str(i) + filename_suffix
                

            print(shape_feat.first().getInfo())
            



            # Run extract function
            simple_extract_function.process_fire_data(
                fireID = fireID,
                fire_list = fire_list,
                local_directory = local_directory,
                wto_outcome = wto_outcome2,
                variable_list = variable_list,
                bucket_name = bucket_name,
                bucket_fp = bucket_fp,
                start_day = user_inputs["start_day_DOY"],
                end_day = user_inputs["end_day_DOY"],
                output_directory = gee_output_directory,
                reducer = ee.Reducer.mean(),
                filename_suffix = filename_suffix2,
                feature_collection = shape_feat, 
                
                mask = mask2)

            processed_fp = fireID + filename_suffix2 + "_df" + ".csv"
            processed_fires.append(processed_fp)
    

{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-122.00249261422391, 43.086121243748835]}, 'id': '0', 'properties': {'MTBS_ID': 'OR4307312203920180715', 'X': 581191.818547496, 'Y': 4770861.30098008, 'latitude': 43.086120551045646, 'longitude': -122.00248944024673, 'point_id': 1, 'response': 167.876543209877, 'response_bin': 0}}
Processing OR4307312203920180715 fire using provided FeatureCollection.
{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-123.57229673716165, 41.99736090042515]}, 'id': '0', 'properties': {'MTBS_ID': 'CA4195612355120180715', 'X': 452601.703147175, 'Y': 4649641.59854937, 'latitude': 41.99736242669278, 'longitude': -123.57229539924559, 'point_id': 1, 'response': 191.296296296296, 'response_bin': 0}}
Processing CA4195612355120180715 fire using provided FeatureCollection.
{'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [-122.54498262165042, 40.93555158023252]}, 'id': '0', 'properties': {'MTBS_ID': 'CA40650122630201

In [26]:

# NOTE: you need to wait for the tasks to finish in GEE and export
# check https://code.earthengine.google.com/tasks to make sure they all are completed before running the code below to download the files to your local computer.

fireIDs = fire_list['MTBS_ID'].values.tolist()

storage_client = storage.Client()
bucket = storage_client.bucket(bucket_name)

# Where to look in the bucket
prefix = f"{bucket_fp}/{gee_output_directory}".strip("/")

# Local output folder
local_output_root = os.path.join(user_inputs["processed_data_directory"], "processed_outputs", gee_output_directory)
os.makedirs(local_output_root, exist_ok=True)

# Control: set True to overwrite existing local files
overwrite = True

found_fireids = set()
downloaded_paths = []

print(f"Listing blobs under: {prefix}")
for blob in bucket.list_blobs(prefix=prefix):
    name = blob.name  # full path inside bucket
    # Only consider files that end with the expected CSV suffix
    if not name.endswith("_initial_pts_df.csv"):
        continue

    # Check if the blob filename contains any of the fireIDs
    matched = None
    for fid in fireIDs:
        if fid in name:
            matched = fid
            break

    if not matched:
        # Not one of the fireIDs of interest
        continue

    found_fireids.add(matched)

    # Local filename (you can preserve paths instead if desired)
    local_path = os.path.join(local_output_root, os.path.basename(name))
    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    if os.path.exists(local_path) and not overwrite:
        print(f"Skipping existing file: {local_path}")
    else:
        print(f"Downloading {name} -> {local_path}")
        blob.download_to_filename(local_path)
        downloaded_paths.append(local_path)

# Summary
missing = sorted(set(fireIDs) - found_fireids)
print(f"\nDownloaded {len(downloaded_paths)} file(s).")
if missing:
    print(f"FireIDs with no matches: {len(missing)} (example: {missing[:10]})")
else:
    print("All fireIDs had at least one matching file.")


Listing blobs under: klamath/processed_outputs/burn_severity/gee_point_extracts

Downloaded 73 file(s).
FireIDs with no matches: 1 (example: ['CA4174812105620190905'])
